This tutorial covers the workflow of a sequence classification project with PyTorch. We'll cover the basics of sequence classification using a simple, but effective, neural bag-of-words model, and how to use the datasets/torchtext libaries to simplify data loading/preprocessing.

Import Modules

In [21]:
import collections

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchtext
from torch._export.db.examples import fn_with_kwargs
from torchtext.data.utils import get_tokenizer
import tqdm

D:\Software\anaconda\envs\nlp\lib\site-packages\torchtext\data\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


In [5]:
seed = 1234
np.random.seed(seed)
torch.manual_seed(seed)

Loading the Dataset

In [18]:
train_data, test_data = datasets.load_dataset('imdb', split=['train', 'test'])

In [12]:
train_data, test_data

(Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }))

In [13]:
train_data.features

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

In [14]:
test_data.features

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

In [15]:
train_data[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

Tokenization

In [22]:
# tokenizer = torchtext.data.utils.get_tokenizer('basic_english')  # from torchtext.data.utils import get_tokenizer
tokenizer = get_tokenizer('basic_english')

In [24]:
tokenizer("Hello World! How are you doing today? i'm doing fantastic!")

['hello',
 'world',
 '!',
 'hello',
 ',',
 'how',
 'are',
 'you',
 'doing',
 'today',
 '?',
 'i',
 "'",
 'm',
 'doing',
 'fantastic',
 '!']

In [27]:
def tokenize_example(example, tokenizer, maxlength):
    tokens = tokenizer(example['text'])[:maxlength]
    return {'tokens': tokens}

In [28]:
tokenize_example(train_data[0], tokenizer, maxlength=200)

{'tokens': ['i',
  'rented',
  'i',
  'am',
  'curious-yellow',
  'from',
  'my',
  'video',
  'store',
  'because',
  'of',
  'all',
  'the',
  'controversy',
  'that',
  'surrounded',
  'it',
  'when',
  'it',
  'was',
  'first',
  'released',
  'in',
  '1967',
  '.',
  'i',
  'also',
  'heard',
  'that',
  'at',
  'first',
  'it',
  'was',
  'seized',
  'by',
  'u',
  '.',
  's',
  '.',
  'customs',
  'if',
  'it',
  'ever',
  'tried',
  'to',
  'enter',
  'this',
  'country',
  ',',
  'therefore',
  'being',
  'a',
  'fan',
  'of',
  'films',
  'considered',
  'controversial',
  'i',
  'really',
  'had',
  'to',
  'see',
  'this',
  'for',
  'myself',
  '.',
  'the',
  'plot',
  'is',
  'centered',
  'around',
  'a',
  'young',
  'swedish',
  'drama',
  'student',
  'named',
  'lena',
  'who',
  'wants',
  'to',
  'learn',
  'everything',
  'she',
  'can',
  'about',
  'life',
  '.',
  'in',
  'particular',
  'she',
  'wants',
  'to',
  'focus',
  'her',
  'attentions',
  'to',
  '

In [29]:
maxlength = 256
train_data = train_data.map(tokenize_example, fn_kwargs={'tokenizer': tokenizer ,'maxlength': maxlength})
test_data = test_data.map(tokenize_example, fn_kwargs={'tokenizer': tokenizer ,'maxlength': maxlength})

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [30]:
train_data

Dataset({
    features: ['text', 'label', 'tokens'],
    num_rows: 25000
})